### Create Table with NP confirmed removed.
ANTI JOIN based on the PTID of matching records to the NP confirmed table

In [0]:
%sql
CREATE OR REPLACE TABLE final_multimodal_adni_living AS
SELECT f.*
FROM final_multimodal_adni f
LEFT ANTI JOIN merged_neuropath_multimodal_adni d
  ON f.PTID = d.PTID

The dataset I downloaded and sent to you is shown in the below cell:

In [0]:
%sql
SELECT *
FROM final_multimodal_adni_living
LIMIT 10;

Total Count (Living)

In [0]:
%sql
SELECT COUNT(*) AS living_count
FROM final_multimodal_adni_living;

Describe table from spark

In [0]:
df = spark.read.table('workspace.default.final_multimodal_adni_living')
display(df.describe())

Check total feature count, 213

In [0]:
%sql
SELECT COUNT(*) AS num_columns
FROM information_schema.columns
WHERE table_name = 'final_multimodal_adni_living';

## Start of Data Quality Check
Checks for:
- Feature Missingness
- Feature Value Dominance

Keep tally on total record count for missingness check

In [0]:
total_rows = df.count()

Null count for missingness check

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

missing_df = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

### Create Missingness Summary
Use column names to sort in output

In [0]:
missing_summary = (
    missing_df
    .toPandas()
    .T
    .reset_index()
)

missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_pct"] = missing_summary["missing_count"] / total_rows
display(missing_summary)

Value Dominance Check

In [0]:
nzv_stats = []

for c in df.columns:
    counts = (
        df.groupBy(c)
          .count()
          .orderBy(col("count").desc())
    )
    
    top = counts.first()
    
    if top is not None:
        nzv_stats.append((
            c,
            counts.count(),           # num of unique values
            top["count"],             # most frequent value count
            top["count"] / total_rows # ratio
        ))


In [0]:
# store as Spark DataFrame and display to see where there are nzv cols
nzv_summary = spark.createDataFrame(
    nzv_stats,
    ["column", "unique_count", "top_count", "top_ratio"]
)

display(nzv_summary.orderBy(col("top_ratio").desc()))

Join tables to see which features have a lot of both
- Change nzv table to pandas
- Join on the column name

In [0]:
nzv_pd = nzv_summary.toPandas()

In [0]:
feature_check = missing_summary.merge(
    nzv_pd,
    on="column",        # merging using the column name
    how="left"
)

An issue arises where we see that the missing count is so high on some that it is the dominant value. We can take this into account when looking into features that we might want to remove before conducting actual feature selection and engineering techniques like correlation or VIF so make sure we aren't wasting time on bad variables with high missingness.

pet_processing and VSHGTSC are variables with 100% NULL values, which we should almost surely remove entirely

As for other values, we should look to remove any where 

In [0]:
display(
    feature_check.sort_values(
        ["missing_pct", "top_ratio"],
        ascending=[True, True]
    )
)